# Notebook 01: Data Loading and Quality Control

**Project:** Cancer Microbiome Comparative Analysis  
**Student:** Claire Chien  
**Description:** This notebook loads microbiome abundance data for Colorectal, Breast, and Prostate cancers, attaches sample labels, performs quality control checks, and saves intermediate data for downstream analysis.

In [ ]:
# ============================================================
# CELL 1 — Import Libraries
# ============================================================
# Think of imports as loading tools from a toolbox before you start a project.
# We need different tools for different jobs below.

import pandas as pd          # pandas: the main tool for working with tables (like Excel in Python)
import numpy as np           # numpy: fast math on arrays and numbers
import matplotlib            # matplotlib: the base library for making charts and graphs
matplotlib.use('Agg')        # 'Agg' means save charts to files instead of popping up windows
import matplotlib.pyplot as plt  # pyplot: the easy-to-use part of matplotlib for drawing plots
import seaborn as sns        # seaborn: makes prettier statistical charts on top of matplotlib
import os                    # os: lets Python talk to your computer's file system (folders, paths)
import warnings              # warnings: controls Python warning messages
warnings.filterwarnings('ignore')  # hide minor warning messages so output stays clean

print('All imports successful.')              # confirm everything loaded without errors
print(f'pandas version: {pd.__version__}')   # show which version of pandas is installed
print(f'numpy version: {np.__version__}')    # show which version of numpy is installed

In [ ]:
# ============================================================
# CELL 2 — Set Up Folder Paths
# ============================================================
# Before we load any data, we tell Python exactly WHERE the files are.
# Think of this as writing down your locker combination before you try to open it.

# os.path.abspath turns a relative path ("../") into a full absolute path on your computer.
# os.getcwd() = "current working directory", i.e., the Notebooks/ folder we're running from.
# Going one level up with ".." reaches the project root: Final_Solution/
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Build the paths for each sub-folder by joining BASE_DIR with the folder name
DATA_DIR    = os.path.join(BASE_DIR, 'Data')      # raw sequencing data lives here
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')   # computed tables will be saved here
FIGURES_DIR = os.path.join(BASE_DIR, 'Figures')   # plots will be saved here

# exist_ok=True means "don't raise an error if the folder already exists"
os.makedirs(RESULTS_DIR, exist_ok=True)  # create Results/ if it doesn't exist yet
os.makedirs(FIGURES_DIR, exist_ok=True)  # create Figures/ if it doesn't exist yet

print(f'BASE_DIR    : {BASE_DIR}')    # confirm the root folder location
print(f'DATA_DIR    : {DATA_DIR}')    # confirm data folder
print(f'RESULTS_DIR : {RESULTS_DIR}') # confirm results folder
print(f'FIGURES_DIR : {FIGURES_DIR}') # confirm figures folder

# --- Dataset-level paths ---
# Each cancer project has its own sub-folder inside Data/, matching the public SRA project ID.
COLORECTAL_DIR = os.path.join(DATA_DIR, 'Colorectal_cancer_project', 'data', 'PRJNA290926')
BREAST_DIR     = os.path.join(DATA_DIR, 'Breast_cancer_project', 'PRJNA658160')
PROSTATE_DIR   = os.path.join(DATA_DIR, 'Prostate_cancer_project', 'data', 'PRJNA1298576')

print('\nDataset directories:')
print(f'  Colorectal : {COLORECTAL_DIR}')  # colorectal cancer samples
print(f'  Breast     : {BREAST_DIR}')      # breast cancer samples
print(f'  Prostate   : {PROSTATE_DIR}')    # prostate cancer samples

In [ ]:
# ============================================================
# CELL 3 — Load Colorectal Cancer Data (PRJNA290926)
# ============================================================
# A "merged abundance table" is a big spreadsheet where each row is one patient sample
# and each column is a different species of bacteria.
# The numbers in the table are "relative abundances" — fractions between 0 and 1
# that tell us what percentage of the bacteria in that sample belongs to each species.

abund_path_cr  = os.path.join(COLORECTAL_DIR, 'merged_abundance_table.csv')  # path to main abundance file
meta_path_cr   = os.path.join(COLORECTAL_DIR, 'filtered_samples.csv')         # path to sample labels file
sra_path_cr    = os.path.join(COLORECTAL_DIR, 'SraRunTable.csv')              # metadata downloaded from NCBI SRA

abund_cr_raw = pd.read_csv(abund_path_cr, index_col=0)   # load abundance table; first column is sample ID (the row index)
meta_cr_raw  = pd.read_csv(meta_path_cr,  index_col=0)   # load metadata; columns include diagnosis (Normal / Cancer)

print('=== Colorectal Raw Shapes ===')
print(f'  Abundance  : {abund_cr_raw.shape}  (samples x species)')   # how many samples and species
print(f'  Metadata   : {meta_cr_raw.shape}')                         # how many samples have metadata
print(f'  Metadata columns: {list(meta_cr_raw.columns)}')            # show available metadata fields

# Some CSV files store the Run ID (sample name) as a column instead of the row index.
# If that happens, we move it to become the index so it matches the abundance table.
if 'Run' in meta_cr_raw.columns:
    meta_cr_raw = meta_cr_raw.set_index('Run')  # promote 'Run' column → row index

# Keep only samples that exist in BOTH the abundance table AND the metadata.
# This removes any sample that was in one file but not the other (a common data issue).
common_idx_cr = abund_cr_raw.index.intersection(meta_cr_raw.index)  # find shared sample IDs
abund_cr_raw  = abund_cr_raw.loc[common_idx_cr]                     # keep only matching rows in abundance
meta_cr_raw   = meta_cr_raw.loc[common_idx_cr, ['diagnosis']]       # keep only the 'diagnosis' column from metadata

# Add two new columns that will be used throughout the project for consistency:
#   cancer_type : which cancer this sample belongs to
#   condition   : 'Cancer' or 'Healthy' (harmonised across all three datasets)
meta_cr_raw['cancer_type'] = 'Colorectal'  # all rows come from this study
meta_cr_raw['condition']   = meta_cr_raw['diagnosis'].map(
    {'Normal': 'Healthy', 'Cancer': 'Cancer'}  # translate original labels to our standard labels
)

print('\n=== Colorectal after label attachment ===')
print(f'  Shape      : {abund_cr_raw.shape}')           # should match abundance after filtering
print('  Condition counts:')
print(meta_cr_raw['condition'].value_counts().to_string(index=True))  # how many cancer vs healthy

# Sanity check: if any diagnosis values were not in our mapping dictionary, they become NaN.
unmapped = meta_cr_raw['condition'].isna().sum()
if unmapped > 0:
    print(f'  WARNING: {unmapped} samples with unmapped diagnosis values:')
    print(meta_cr_raw[meta_cr_raw['condition'].isna()]['diagnosis'].value_counts())

In [ ]:
# ============================================================
# CELL 4 — Load Breast Cancer Data (PRJNA658160)
# ============================================================
# The breast cancer dataset uses a different file format than colorectal.
# The abundance file already has genus-level data (not species-level).
# The case/control labels are stored in a separate SraRunTable.csv from NCBI.

abund_path_br = os.path.join(BREAST_DIR, 'kraken_G_abundance_matrix.csv')  # genus-level abundance file
sra_path_br   = os.path.join(BREAST_DIR, 'SraRunTable.csv')                # sample metadata from NCBI SRA

abund_br_raw = pd.read_csv(abund_path_br, index_col=0)  # load abundance: rows=samples, cols=genera
sra_br       = pd.read_csv(sra_path_br)                 # load SraRunTable: contains case/control column

print('=== Breast Raw Shapes ===')
print(f'  Abundance     : {abund_br_raw.shape}  (samples x genera)')  # each column is a genus of bacteria
print(f'  SraRunTable   : {sra_br.shape}')                           # full metadata table from NCBI
print(f'  SraRunTable columns (first 10): {list(sra_br.columns[:10])}')  # preview of available columns

# The SraRunTable's 'Run' column holds the sample IDs.  We make it the row index
# so we can match rows to the abundance table by sample name.
if 'Run' in sra_br.columns:
    sra_br = sra_br.set_index('Run')  # 'Run' column becomes the row index
else:
    # Some versions of SraRunTable use a different column name — try to find it automatically
    run_col = [c for c in sra_br.columns if 'run' in c.lower() or 'accession' in c.lower()]
    if run_col:
        sra_br = sra_br.set_index(run_col[0])
        print(f'  Used column "{run_col[0]}" as Run index')

# Different NCBI tables name the cancer/healthy column differently.
# We check several possible names and use the first one we find.
cc_col = None
for candidate in ['case_control', 'Case_Control', 'CaseControl', 'sample_type', 'condition', 'phenotype']:
    if candidate in sra_br.columns:
        cc_col = candidate   # found the column with our labels
        break
if cc_col is None:
    print('  WARNING: could not find case_control column. Columns are:')
    print(list(sra_br.columns))
else:
    print(f'  Using column "{cc_col}" for case/control labels')
    print(f'  Unique values: {sra_br[cc_col].unique()}')  # show what label values exist

# Keep only samples present in BOTH abundance and SraRunTable
common_idx_br = abund_br_raw.index.intersection(sra_br.index)  # find overlapping sample IDs
abund_br_raw  = abund_br_raw.loc[common_idx_br]                 # filter abundance rows
meta_br_raw   = sra_br.loc[common_idx_br, [cc_col]].copy() if cc_col else sra_br.loc[common_idx_br].copy()

# Add standard 'cancer_type' and 'condition' columns (same as we did for colorectal)
meta_br_raw['cancer_type'] = 'Breast'  # tag every sample as Breast cancer
if cc_col:
    # Map various possible spellings of "Case" and "Control" to our standard labels
    meta_br_raw['condition'] = meta_br_raw[cc_col].map(
        {'Case': 'Cancer', 'Control': 'Healthy',
         'case': 'Cancer', 'control': 'Healthy',
         'CASE': 'Cancer', 'CONTROL': 'Healthy'}
    )
else:
    meta_br_raw['condition'] = 'Unknown'  # fallback if we couldn't find the label column

print('\n=== Breast after label attachment ===')
print(f'  Shape      : {abund_br_raw.shape}')
print('  Condition counts:')
print(meta_br_raw['condition'].value_counts().to_string())  # Cancer vs Healthy sample counts

unmapped_br = meta_br_raw['condition'].isna().sum()
if unmapped_br > 0:
    print(f'  WARNING: {unmapped_br} samples with unmapped labels')

In [ ]:
# ============================================================
# CELL 5 — Load Prostate Cancer Data (PRJNA1298576)
# ============================================================
# Important limitation: this prostate dataset has NO healthy (control) samples.
# All 31 samples are cancer biopsies.  We can still classify them by cancer type
# (Colorectal vs Breast vs Prostate), but we cannot do cancer vs healthy comparison.

abund_path_pr = os.path.join(PROSTATE_DIR, 'kraken_G_abundance_matrix.csv')  # genus-level abundance
sra_path_pr   = os.path.join(PROSTATE_DIR, 'SraRunTable.csv')                # NCBI sample metadata

abund_pr_raw = pd.read_csv(abund_path_pr, index_col=0)  # rows=samples, cols=genera
sra_pr       = pd.read_csv(sra_path_pr)                 # metadata (all rows are cancer)

print('=== Prostate Raw Shapes ===')
print(f'  Abundance     : {abund_pr_raw.shape}  (samples x genera)')
print(f'  SraRunTable   : {sra_pr.shape}')

# Make 'Run' the row index so it aligns with the abundance table
if 'Run' in sra_pr.columns:
    sra_pr = sra_pr.set_index('Run')
else:
    run_col = [c for c in sra_pr.columns if 'run' in c.lower() or 'accession' in c.lower()]
    if run_col:
        sra_pr = sra_pr.set_index(run_col[0])
        print(f'  Used column "{run_col[0]}" as Run index')

# Keep only samples present in both files
common_idx_pr = abund_pr_raw.index.intersection(sra_pr.index)
abund_pr_raw  = abund_pr_raw.loc[common_idx_pr]
meta_pr_raw   = sra_pr.loc[common_idx_pr].copy()

# Tag every prostate sample as 'Cancer' — there is no healthy control group
meta_pr_raw['cancer_type'] = 'Prostate'
meta_pr_raw['condition']   = 'Cancer'  # hardcoded because no controls exist in this study

print('\n=== Prostate after label attachment ===')
print(f'  Shape      : {abund_pr_raw.shape}')
print('  Condition counts:')
print(meta_pr_raw['condition'].value_counts().to_string())
print('\n  LIMITATION NOTE: PRJNA1298576 contains only prostate biopsy samples.')
print('  There is no matched control (healthy) group in this dataset.')
print('  All samples are labelled as Cancer for downstream comparative analysis.')

In [ ]:
# ============================================================
# CELL 6 — Quality Control (QC) Filtering
# ============================================================
# Before doing any analysis, we check that our data is "clean":
#   1. No missing values (NaN)
#   2. No samples that are completely empty (all zeros)
#   3. Each sample's bacteria percentages add up to ~1.0 (i.e. 100%)
#   4. No samples with suspiciously low total counts (very low library size)
#
# Think of QC like checking your ingredients before baking — bad data in = bad results out.

def run_qc(abund_df, meta_df, dataset_name, row_sum_tol=0.01, min_lib_size=0.5):
    """
    Perform QC on a relative-abundance DataFrame.

    Parameters
    ----------
    abund_df      : pd.DataFrame — samples x taxa, values in [0, 1]
    meta_df       : pd.DataFrame — metadata aligned to abund_df
    dataset_name  : str
    row_sum_tol   : float — tolerance around 1.0 for row-sum check
    min_lib_size  : float — minimum acceptable row sum (flag if below)

    Returns
    -------
    abund_clean, meta_clean : filtered DataFrames
    """
    print(f'\n{"="*60}')
    print(f'QC Report: {dataset_name}')
    print(f'{"="*60}')
    print(f'  Input shape : {abund_df.shape}')  # how many samples and features we start with

    # --- Check 1: Missing values ---
    # NaN means "no data" — this can happen if a sample was not sequenced for a given species.
    n_missing = abund_df.isna().sum().sum()          # count total NaN cells in the entire table
    print(f'  Missing values (NaN): {n_missing}')
    if n_missing > 0:
        abund_df = abund_df.fillna(0)                # replace NaN with 0 (bacteria not detected)
        print(f'    -> Filled {n_missing} NaN values with 0')

    # --- Check 2: Empty samples ---
    # A sample with all zeros was never successfully sequenced — no useful data.
    row_sums = abund_df.sum(axis=1)                  # sum each row (sample) across all bacteria columns
    empty_mask = row_sums == 0                       # True for rows that are completely zero
    n_empty = empty_mask.sum()
    print(f'  Empty samples (all-zero rows): {n_empty}')
    if n_empty > 0:
        abund_df = abund_df.loc[~empty_mask]         # remove empty sample rows
        meta_df  = meta_df.loc[~empty_mask]          # remove the matching metadata rows too
        print(f'    -> Dropped {n_empty} empty samples')
        row_sums = abund_df.sum(axis=1)              # recompute row sums after removing empty rows

    # --- Check 3: Row-sum check (should be ~1.0) ---
    # For relative abundance, all percentages in a sample should sum to exactly 1 (= 100%).
    # We allow a tiny tolerance of ±1% to account for rounding.
    outlier_mask = (row_sums < (1.0 - row_sum_tol)) | (row_sums > (1.0 + row_sum_tol))
    n_outlier = outlier_mask.sum()
    print(f'  Samples with row sum outside [1 ± {row_sum_tol}]: {n_outlier}')
    if n_outlier > 0:
        print(f'    Row sum stats: min={row_sums.min():.4f}, max={row_sums.max():.4f}, '
              f'mean={row_sums.mean():.4f}')

    # --- Check 4: Low library size ---
    # If a sample's total is below 0.5, it might represent an extremely sparse / failed sequencing run.
    # We flag these but do not remove them automatically.
    low_lib_mask = row_sums < min_lib_size
    n_low = low_lib_mask.sum()
    print(f'  Samples with row sum < {min_lib_size}: {n_low}')
    if n_low > 0:
        print(f'    -> These samples may be low quality; flagging but NOT dropping here.')
        print(f'    -> Low-lib samples: {list(abund_df.index[low_lib_mask])}')

    print(f'  Output shape: {abund_df.shape}')  # final shape after QC
    return abund_df, meta_df  # return cleaned abundance and matching metadata


# Run QC on all three datasets
abund_cr_qc, meta_cr_qc = run_qc(abund_cr_raw.copy(), meta_cr_raw.copy(), 'Colorectal (PRJNA290926)')
abund_br_qc, meta_br_qc = run_qc(abund_br_raw.copy(), meta_br_raw.copy(), 'Breast (PRJNA658160)')
abund_pr_qc, meta_pr_qc = run_qc(abund_pr_raw.copy(), meta_pr_raw.copy(), 'Prostate (PRJNA1298576)')

print('\nQC complete for all three datasets.')

In [ ]:
# ============================================================
# CELL 7 — Separate Metadata from Abundance Tables
# ============================================================
# After QC, we split each dataset into two clean pieces:
#   abund_* : numbers only (bacteria abundance values)
#   meta_*  : labels only (cancer_type and condition columns)
# Having them separate makes downstream analysis cleaner.

META_COLS = ['condition', 'cancer_type']  # the only two label columns we need going forward

# --- Colorectal ---
meta_colorectal  = meta_cr_qc[META_COLS].copy()   # extract just the two label columns
abund_colorectal = abund_cr_qc.copy()              # copy the bacteria abundance data
# Safety: remove any label columns that accidentally ended up in the abundance table
abund_colorectal = abund_colorectal.drop(
    columns=[c for c in META_COLS if c in abund_colorectal.columns], errors='ignore'
)

# --- Breast ---
# meta_br_qc may still have many extra columns from SraRunTable — we only want META_COLS.
extra_br_cols = [c for c in meta_br_qc.columns if c not in META_COLS]  # columns we don't need
meta_breast  = meta_br_qc[META_COLS].copy()   # keep only our two label columns
abund_breast = abund_br_qc.copy()
abund_breast = abund_breast.drop(
    columns=[c for c in META_COLS if c in abund_breast.columns], errors='ignore'
)

# --- Prostate ---
meta_prostate  = meta_pr_qc[META_COLS].copy()   # all prostate samples are Cancer
abund_prostate = abund_pr_qc.copy()
abund_prostate = abund_prostate.drop(
    columns=[c for c in META_COLS if c in abund_prostate.columns], errors='ignore'
)

print('=== Separated DataFrames ===')
print(f'abund_colorectal  : {abund_colorectal.shape}  (samples x species)')  # bacteria columns
print(f'meta_colorectal   : {meta_colorectal.shape}')                        # label columns
print(f'abund_breast      : {abund_breast.shape}  (samples x genera)')
print(f'meta_breast       : {meta_breast.shape}')
print(f'abund_prostate    : {abund_prostate.shape}  (samples x genera)')
print(f'meta_prostate     : {meta_prostate.shape}')

print('\nMetadata preview (Colorectal):')
print(meta_colorectal.head(3))   # show first 3 rows to confirm format
print('\nMetadata preview (Breast):')
print(meta_breast.head(3))
print('\nMetadata preview (Prostate):')
print(meta_prostate.head(3))

In [ ]:
# ============================================================
# CELL 8 — QC Figure: Sample Count Bar Chart
# ============================================================
# Before analysis, it's important to visualize HOW MANY samples we have
# in each group.  Imbalanced groups (many cancer but few healthy) can
# affect statistical tests and machine learning accuracy.

cancer_types = ['Colorectal', 'Breast', 'Prostate']  # the three cancer datasets
metas = [meta_colorectal, meta_breast, meta_prostate]  # corresponding metadata tables

# Count how many Cancer and Healthy samples exist for each cancer type
counts = {}
for ct, meta in zip(cancer_types, metas):
    vc = meta['condition'].value_counts()  # value_counts() counts how many times each label appears
    counts[ct] = {
        'Cancer' : vc.get('Cancer',  0),   # get count for 'Cancer'; default 0 if not found
        'Healthy': vc.get('Healthy', 0)    # get count for 'Healthy'; default 0 if not found
    }

counts_df = pd.DataFrame(counts).T   # transpose so rows=cancer types, columns=Cancer/Healthy
print('Sample counts by cancer type and condition:')
print(counts_df)

# --- Build a stacked bar chart ---
fig, ax = plt.subplots(figsize=(7, 5))  # create a 7×5 inch figure

x = np.arange(len(cancer_types))  # positions for the bars on the x-axis: [0, 1, 2]
bar_width = 0.55                   # width of each bar (fraction of spacing between bars)

# Draw the Cancer bars (bottom layer of the stack)
bars_cancer  = ax.bar(x, counts_df['Cancer'],  bar_width,
                      label='Cancer',  color='#C0392B', edgecolor='white')  # red bars

# Draw the Healthy bars stacked ON TOP of the Cancer bars
bars_healthy = ax.bar(x, counts_df['Healthy'], bar_width,
                      bottom=counts_df['Cancer'],         # start each bar where Cancer bar ends
                      label='Healthy', color='#2980B9', edgecolor='white')  # blue bars

# Add white number labels inside the Cancer portion of each bar
for bar, val in zip(bars_cancer, counts_df['Cancer']):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2,  # horizontal center of the bar
                bar.get_height() / 2,                # vertical center of the Cancer portion
                str(int(val)),                        # the count as a string
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Add white number labels inside the Healthy portion of each bar
for bar, bottom_val, val in zip(bars_healthy, counts_df['Cancer'], counts_df['Healthy']):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bottom_val + val / 2,   # place the label in the middle of the Healthy portion
                str(int(val)),
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cancer_types, fontsize=12)   # label the x-axis with cancer type names
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Sample Counts by Cancer Type and Condition', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, counts_df.sum(axis=1).max() * 1.15)  # add 15% space above tallest bar
sns.despine(ax=ax)  # remove top and right border lines for a cleaner look

fig_path = os.path.join(FIGURES_DIR, 'fig00_sample_counts.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')  # save at 150 DPI
plt.close()                                            # close figure to free memory
print(f'Saved: {fig_path}')

In [ ]:
# ============================================================
# CELL 9 — QC Figure: Row-Sum Distribution Histograms
# ============================================================
# For relative abundance data, each sample's values should sum to exactly 1.0 (100%).
# This histogram lets us visually check that all samples pass this test.
# A pile of values at 1.0 means everything is correct.
# Bars far from 1.0 would signal a data problem.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 3 side-by-side panels for the 3 datasets

# List of (data, title, color) tuples to loop over
datasets = [
    (abund_colorectal, 'Colorectal\n(PRJNA290926)', '#27AE60'),  # green
    (abund_breast,     'Breast\n(PRJNA658160)',     '#8E44AD'),  # purple
    (abund_prostate,   'Prostate\n(PRJNA1298576)',  '#E67E22'),  # orange
]

for ax, (abund_df, title, color) in zip(axes, datasets):
    row_sums = abund_df.sum(axis=1)             # compute each sample's total (should be ~1.0)
    ax.hist(row_sums, bins=30, color=color, edgecolor='white', alpha=0.85)  # draw the histogram
    ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Expected sum = 1')  # ideal line
    ax.axvline(row_sums.mean(), color='black', linestyle='-', linewidth=1.2,
               label=f'Mean = {row_sums.mean():.3f}')  # actual mean line
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Row Sum (relative abundance)', fontsize=10)  # x-axis label
    ax.set_ylabel('Number of Samples', fontsize=10)             # y-axis label
    ax.legend(fontsize=9)
    sns.despine(ax=ax)  # clean up the chart borders

fig.suptitle('Row-Sum Distribution per Dataset (QC Check)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'fig00_row_sum_qc.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {fig_path}')

# Print exact min/max/mean for each dataset so we can verify there are no outliers
print('\nRow-sum summary statistics:')
for abund_df, title, _ in datasets:
    rs = abund_df.sum(axis=1)
    label = title.replace('\n', ' ')
    print(f'  {label:30s}  min={rs.min():.4f}  max={rs.max():.4f}  mean={rs.mean():.4f}')

In [ ]:
# ============================================================
# CELL 10 — Save Processed Data to Results/
# ============================================================
# Now that each dataset is loaded, labelled, and passed QC, we save them to CSV files.
# The next notebooks (02, 03, 04, ...) will load from these files instead of re-reading
# the raw data — this keeps the pipeline clean and reproducible.

# --- Save abundance tables (bacteria data) ---
abund_colorectal.to_csv(os.path.join(RESULTS_DIR, 'abund_colorectal.csv'))  # 268 samples × 1616 species
abund_breast.to_csv(    os.path.join(RESULTS_DIR, 'abund_breast.csv'))      # 319 samples × 679 genera
abund_prostate.to_csv(  os.path.join(RESULTS_DIR, 'abund_prostate.csv'))    # 31 samples × 844 genera

# --- Save metadata tables (labels) ---
meta_colorectal.to_csv(os.path.join(RESULTS_DIR, 'meta_colorectal.csv'))   # condition + cancer_type
meta_breast.to_csv(    os.path.join(RESULTS_DIR, 'meta_breast.csv'))
meta_prostate.to_csv(  os.path.join(RESULTS_DIR, 'meta_prostate.csv'))

print('=== Saved to Results/ ===')
saved_files = [
    ('abund_colorectal.csv',  abund_colorectal.shape),   # (rows, columns)
    ('abund_breast.csv',      abund_breast.shape),
    ('abund_prostate.csv',    abund_prostate.shape),
    ('meta_colorectal.csv',   meta_colorectal.shape),
    ('meta_breast.csv',       meta_breast.shape),
    ('meta_prostate.csv',     meta_prostate.shape),
]
for fname, shape in saved_files:
    print(f'  {fname:35s}  shape={shape}')  # confirm each file was written with expected size

print('\nNotebook 01 complete. Proceed to 02_taxonomic_harmonization.ipynb.')